# 13 · Model solution: market direction

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lukasquaderer-lgtm/Linear-regression/blob/master/notebooks/solutions/13_project_market_direction_solution.ipynb)

One reasonable solution. Spoiler: the evidence for predictability is weak, which is the most common honest answer
in return forecasting.

In [1]:
# Setup: run this cell first (Shift + Enter). Works in Jupyter, VS Code and Google Colab.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # hide library deprecation notices

# Use the local data folder if it exists, otherwise read the CSVs straight from GitHub (e.g. in Colab)
DATA = next((p for p in ["../data/", "../../data/"] if Path(p).exists()), "https://raw.githubusercontent.com/lukasquaderer-lgtm/Linear-regression/master/data/")
pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (8, 4)

# Answer checker: after an exercise, check("01.1") tells you whether your answer is right
import sys
if Path("../checks.py").exists():
    sys.path.insert(0, "..")  # solution notebooks live one folder down
try:
    from checks import check, check_all
except ImportError:  # in Colab: fetch the checker from GitHub
    import urllib.request
    urllib.request.urlretrieve("https://raw.githubusercontent.com/lukasquaderer-lgtm/Linear-regression/master/notebooks/checks.py", "checks.py")
    from checks import check, check_all
print("Ready. Data folder:", DATA)

Ready. Data folder: ../../data/


In [2]:
# Load a real dataset from the ISLP package (Introduction to Statistical Learning with Python)
import subprocess, sys
def load_islp(name):
    try:
        import ISLP
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "ISLP"], check=True)
        import ISLP
    return pd.read_csv(Path(ISLP.__file__).parent / "data" / f"{name}.csv")

In [3]:
from scipy.stats import binomtest
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

wk = load_islp("Weekly"); wk["up"] = (wk["Direction"] == "Up").astype(int)
tr, te = wk[wk["Year"] <= 2008], wk[wk["Year"] >= 2009]
always_up = te["up"].mean()
print(f"1) test weeks: {len(te)}, share up: {always_up:.3f} → 'always Up' accuracy")

cols = ["Lag1", "Lag2", "Lag3", "Lag4", "Lag5", "Volume"]
full = sm.Logit(tr["up"], sm.add_constant(tr[cols])).fit(disp=0)
print(full.summary().tables[1])
p = full.predict(sm.add_constant(te[cols])); pred = (p > 0.5).astype(int)
acc = (pred == te["up"]).mean()
print(f"2) test accuracy {acc:.3f}"); print(confusion_matrix(te["up"], pred))

1) test weeks: 104, share up: 0.587 → 'always Up' accuracy
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.3326      0.094      3.530      0.000       0.148       0.517
Lag1          -0.0623      0.029     -2.123      0.034      -0.120      -0.005
Lag2           0.0447      0.030      1.499      0.134      -0.014       0.103
Lag3          -0.0155      0.029     -0.524      0.600      -0.073       0.042
Lag4          -0.0311      0.029     -1.064      0.287      -0.088       0.026
Lag5          -0.0377      0.029     -1.291      0.197      -0.095       0.020
Volume        -0.0897      0.054     -1.658      0.097      -0.196       0.016
2) test accuracy 0.462
[[31 12]
 [44 17]]


In [4]:
def binom_p(correct_pred):
    return binomtest(int(correct_pred.sum()), len(correct_pred), always_up, alternative="greater").pvalue

print(f"3) full model: accuracy {acc:.3f} vs {always_up:.3f}, one-sided binomial p = {binom_p(pred == te['up']):.3f}")

variants = {
    "Logit Lag2": (LogisticRegression(), ["Lag2"]),
    "LDA Lag2": (LinearDiscriminantAnalysis(), ["Lag2"]),
    "QDA Lag2": (QuadraticDiscriminantAnalysis(), ["Lag2"]),
    **{f"KNN k={k} Lag2": (KNeighborsClassifier(k), ["Lag2"]) for k in [1, 5, 25, 75]},
    "Logit Lag1-2": (LogisticRegression(), ["Lag1", "Lag2"]),
    "Random forest all": (RandomForestClassifier(300, min_samples_leaf=20, random_state=0), cols),
}
rows = []
for name, (m, c) in variants.items():
    pr = m.fit(tr[c], tr["up"]).predict(te[c]); ok = pr == te["up"]
    rows.append([name, ok.mean(), binom_p(ok)])
tab = pd.DataFrame(rows, columns=["model", "test accuracy", "binomial p"]).set_index("model").sort_values("test accuracy", ascending=False)
print("4)"); print(tab.round(3))
print(f"Tried {len(tab)} variants; the best p-value ignores that we picked it from {len(tab)} tries (Bonferroni threshold ≈ {0.05/len(tab):.4f})")

3) full model: accuracy 0.462 vs 0.587, one-sided binomial p = 0.996


4)
                   test accuracy  binomial p
model                                       
Logit Lag2                 0.625       0.244
LDA Lag2                   0.625       0.244
QDA Lag2                   0.587       0.542
Logit Lag1-2               0.577       0.619
KNN k=75 Lag2              0.558       0.758
KNN k=25 Lag2              0.548       0.815
Random forest all          0.548       0.815
KNN k=5 Lag2               0.538       0.863
KNN k=1 Lag2               0.490       0.981
Tried 9 variants; the best p-value ignores that we picked it from 9 tries (Bonferroni threshold ≈ 0.0056)


In [5]:
# 5) does the best-looking model make money?
best = LogisticRegression().fit(tr[["Lag2"]], tr["up"]).predict(te[["Lag2"]])
r = te["Today"].to_numpy() / 100
strat = np.where(best == 1, r, 0.0)
def perf(x):
    return pd.Series({"total return": np.prod(1 + x) - 1, "ann. vol": x.std() * np.sqrt(52), "Sharpe (rf=0)": x.mean() / x.std() * np.sqrt(52)})
print("5)"); print(pd.DataFrame({"Logit Lag2 strategy": perf(strat), "Buy and hold": perf(r)}).round(3))

# 6) walk-forward: re-train each year on all prior data
acc_by_year = {}
for yr in range(2000, 2011):
    trn, tst = wk[wk["Year"] < yr], wk[wk["Year"] == yr]
    m = LogisticRegression().fit(trn[["Lag2"]], trn["up"])
    acc_by_year[yr] = ((m.predict(tst[["Lag2"]]) == tst["up"]).mean(), tst["up"].mean())
wf = pd.DataFrame(acc_by_year, index=["model accuracy", "always-Up accuracy"]).T
print("6)"); print(wf.round(3)); print("Years the model beat 'always Up':", int((wf.iloc[:, 0] > wf.iloc[:, 1]).sum()), "of", len(wf))

5)
               Logit Lag2 strategy  Buy and hold
total return                 0.366         0.350
ann. vol                     0.182         0.216
Sharpe (rf=0)                0.949         0.804
6)
      model accuracy  always-Up accuracy
2000           0.423               0.442
2001           0.442               0.481
2002           0.385               0.423
2003           0.673               0.654
2004           0.538               0.538
2005           0.577               0.577
2006           0.558               0.558
2007           0.547               0.547
2008           0.442               0.442
2009           0.558               0.558
2010           0.673               0.615
Years the model beat 'always Up': 2 of 11


### 7) Memo (example)
The full model's coefficients are mostly insignificant (only Lag2 is borderline), and its test accuracy does not
significantly beat simply predicting “Up” every week. The best variant we found (logistic regression on Lag2) looks
better, but it was chosen after trying about ten models on the same two-year test set, so its apparent edge is
overstated; even before any multiple-testing adjustment its p-value is about 0.24, so it is not significant.
Walk-forward re-training beat “always Up” in only 2 of 11 years. The Lag2 strategy did slightly better than buy-and-hold
over 2009–10 (about 37% vs 35%, with lower volatility), but two years is far too short to separate skill from luck. **Recommendation: do not allocate capital.** If the idea is pursued, pre-register
one model and test it on new data.